# 02 - Fine-tune transformers

**Goal:** fine-tune MuRIL (and optionally mBERT) and compare against the baseline.

**Before running on Kaggle**
1. Turn on a GPU (T4) and Internet in the notebook settings.
2. Add your Hugging Face token under *Add-ons > Secrets* with the label `HF_TOKEN`.
3. Set `HF_USER` below.
4. Run once with `SMOKE = True` (about 2 minutes) to make sure everything works.
5. Set `SMOKE = False`, then use **Save Version > Save & Run All (Commit)** so the outputs are kept.

In [ ]:
%pip install -q ftfy tweet-preprocessor

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/23f2004336/hinglish-sentiment.git"

# Make `src/` importable: works locally (inside notebooks/ or the repo root) and on Kaggle.
if os.path.exists("../src"):
    sys.path.insert(0, "..")
    RESULTS_DIR = "../results"
elif os.path.exists("src"):
    sys.path.insert(0, ".")
    RESULTS_DIR = "results"
else:
    if not os.path.exists("hinglish-sentiment"):
        subprocess.run(["git", "clone", REPO_URL], check=True)   # needs Internet ON in Kaggle
    sys.path.insert(0, os.path.abspath("hinglish-sentiment"))
    RESULTS_DIR = "."

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
import torch
from src.data import load_data
from src.train import run

HF_USER = "YOUR-HF-USERNAME"      # <-- change this
OUT_ROOT = "/kaggle/working" if os.path.exists("/kaggle") else "../outputs"
os.makedirs(OUT_ROOT, exist_ok=True)

print("GPU available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
dfs = load_data()

## MuRIL

In [ ]:
SMOKE = True    # True = quick test on 3000 rows, 1 epoch, nothing pushed. Set to False for the real run.

res_muril = run(
    dfs,
    name="google/muril-base-cased",
    tag="muril",
    epochs=1 if SMOKE else 2,
    out_root=OUT_ROOT,
    repo=None if SMOKE else f"{HF_USER}/hinglish-sentiment-muril",
    max_train=3000 if SMOKE else None,
)

## mBERT (optional)
You already have its numbers from the first run (see the table below). Only rerun if you want to reproduce them.

In [ ]:
RUN_MBERT = False
if RUN_MBERT:
    res_mbert = run(dfs, name="bert-base-multilingual-cased", tag="mbert", epochs=3, out_root=OUT_ROOT)

## Results table

In [ ]:
import pandas as pd

rows = {
    "TF-IDF + LogReg": {"overall": 0.817, "hinglish": 0.518, "english": 0.826},
    "mBERT (3 epochs)": {"overall": 0.847, "hinglish": 0.630, "english": 0.855},   # from the first run
    "MuRIL": {k: res_muril[k] for k in ["overall", "hinglish", "english"]},
}
table = pd.DataFrame(rows).T.round(3)
table

## Notes
- Test scores are looked at once, after training. Choices (best epoch) are made on the validation set.
- If the SMOKE run was used, the MuRIL row above is meaningless. Rerun with `SMOKE = False`.